# 06 – Publish weights + deploy the browser app
1. Uploads the two LoRA adapters to the Hugging Face Hub (public download link for the report).
2. Deploys the app as a free Hugging Face Space (CPU), so an evaluator only needs a browser.
3. Optional: runs the app here on the T4 with a temporary public link (fast; use it to record the demo).

Needs a Colab secret **HF_TOKEN**: huggingface.co → Settings → Access Tokens → *Write* token.

In [ ]:
# ==== EDIT THESE TWO LINES ====
GITHUB_USER = "FatimaRana50"
GIT_NAME    = "Fatima Rana"
# ==============================
REPO = "trialmatch"

!pip -q install peft datasets
!pip -q uninstall -y torchao       # Colab ships torchao 0.10; peft refuses to load with it (we do not use it)
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
token = userdata.get('GH_TOKEN')          # Colab sidebar -> key icon -> add secret GH_TOKEN
if not os.path.exists(f'/content/{REPO}'):
    !git clone -q https://{token}@github.com/{GITHUB_USER}/{REPO}.git /content/{REPO}
%cd /content/{REPO}
!git pull -q
!git config user.name "{GIT_NAME}"
!git config user.email "{GITHUB_USER}@users.noreply.github.com"
import sys; sys.path.insert(0, '/content/trialmatch/src')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
HF_USER = "your-hf-username"          # <- your Hugging Face username
TARGET_DIR = "/content/drive/MyDrive/trialmatch/lora_r16/final_adapter"
DRAFT_DIR  = "/content/drive/MyDrive/trialmatch/draft_0.5b/final_adapter"
TARGET_REPO = f"{HF_USER}/trialmatch-qwen2.5-1.5b-lora"
DRAFT_REPO  = f"{HF_USER}/trialmatch-qwen2.5-0.5b-draft-lora"
SPACE_REPO  = f"{HF_USER}/trialmatch"
!pip -q install -U huggingface_hub
from huggingface_hub import HfApi
api = HfApi(token=userdata.get("HF_TOKEN"))
print("logged in as", api.whoami()["name"])

## 1. Upload adapters

In [ ]:
card = lambda base, what: f"""---
base_model: {base}
library_name: peft
license: apache-2.0
tags: [lora, clinical-trials, trialgpt]
---
# {what}
LoRA adapter for `{base}` trained on the training split (37 patients) of
`ncbi/TrialGPT-Criterion-Annotations` (patient-grouped split, seed 42).
Code, split manifest and evaluation: https://github.com/{GITHUB_USER}/trialmatch
**Educational screening prototype. Not for clinical use.**
"""
for d, repo, base, what in [(TARGET_DIR, TARGET_REPO, "Qwen/Qwen2.5-1.5B-Instruct", "TrialMatch criterion classifier (target)"),
                            (DRAFT_DIR, DRAFT_REPO, "Qwen/Qwen2.5-0.5B-Instruct", "TrialMatch draft model for speculative decoding")]:
    api.create_repo(repo, exist_ok=True)
    open(f"{d}/README.md", "w").write(card(base, what))
    api.upload_folder(folder_path=d, repo_id=repo, commit_message="Upload LoRA adapter")
    print("https://huggingface.co/" + repo)

## 2. Deploy the Space (free CPU; first start takes ~5 min to build and download the model)

In [ ]:
import gradio, shutil, subprocess
GR = gradio.__version__
space_readme = f"""---
title: TrialMatch
emoji: 🩺
colorFrom: blue
colorTo: green
sdk: gradio
sdk_version: {GR}
app_file: app/app.py
pinned: false
---
Educational clinical-trial screening prototype. Source: https://github.com/{GITHUB_USER}/trialmatch
"""
stage = "/content/space"; shutil.rmtree(stage, ignore_errors=True)
for d in ["app", "src", "data"]:
    shutil.copytree(d, f"{stage}/{d}", ignore=shutil.ignore_patterns("__pycache__", "*.ipynb"))
os.makedirs(f"{stage}/results")
for f in ["kv_bench.json", "spec_bench.json", "attention_bench.json", "base_test_metrics.json", "lora_test_metrics.json"]:
    shutil.copy(f"results/{f}", f"{stage}/results/{f}")
open(f"{stage}/README.md", "w").write(space_readme)
open(f"{stage}/requirements.txt", "w").write("\n".join(["torch", "transformers>=4.56", "peft>=0.13", "accelerate", "numpy"]) + "\n")
api.create_repo(SPACE_REPO, repo_type="space", space_sdk="gradio", exist_ok=True)
commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
for k, v in {"TRIALMATCH_ADAPTER": TARGET_REPO, "TRIALMATCH_DRAFT_ADAPTER": DRAFT_REPO, "TRIALMATCH_COMMIT": commit}.items():
    api.add_space_variable(SPACE_REPO, k, v)
api.upload_folder(folder_path=stage, repo_id=SPACE_REPO, repo_type="space", commit_message=f"Deploy {commit}")
print("Space: https://huggingface.co/spaces/" + SPACE_REPO)

## 3. (Optional) Run the app on this GPU with a temporary public link
Open the `https://....gradio.live` link printed below. Stop the cell to shut it down.

In [ ]:
!pip -q install gradio
%env TRIALMATCH_ADAPTER=/content/drive/MyDrive/trialmatch/lora_r16/final_adapter
%env TRIALMATCH_DRAFT_ADAPTER=/content/drive/MyDrive/trialmatch/draft_0.5b/final_adapter
%env TRIALMATCH_SHARE=1
!python app/app.py